# 01 — Data Exploration & Text Preprocessing

**Goal:** Load the equipment inspection dataset, understand its structure, and build the text cleaning pipeline.

**Key decisions documented:**
- Equipment code removal (regex pattern design)
- Stopword removal experiment (why we kept them)
- Dataset statistics and column analysis

## 1. Library Imports

Core data science libraries plus domain-specific NLP tools.

**Selection rationale:**
- `sentence_transformers`: State-of-the-art sentence embeddings
- `AgglomerativeClustering`: Hierarchical — no need to pre-specify K
- `TSNE`: 384-dim → 2D visualization
- `nltk`: Stopword removal + tokenization

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

print(f'pandas: {pd.__version__}')
print(f'numpy: {np.__version__}')
print(f'nltk: {nltk.__version__}')

## 2. NLTK Data Setup (Offline Config)

Download NLTK data to a local directory for portability.
This enables offline operation in corporate/air-gapped environments.

In [ ]:
# Download to local directory for offline portability
nltk.download('punkt', download_dir='./nltk_data')
nltk.download('stopwords', download_dir='./nltk_data')
nltk.data.path.append('./nltk_data')
print('✅ NLTK data ready')

## 3. Data Loading & Inspection

Load the cleaned dataset from the `data/` directory.

In [ ]:
df = pd.read_csv('../data/raw/dataset3.csv')
print(f'Dataset shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head()

In [ ]:
# Basic statistics
print('=== Priority Distribution ===')
print(df['PRIORITY'].value_counts().sort_index())
print(f"\n=== Sample Descriptions ===")
for i, desc in enumerate(df['DESCRIPTION'].sample(5, random_state=42)):
    print(f"{i+1}. {desc[:120]}..." if len(str(desc)) > 120 else f"{i+1}. {desc}")

## 4. Text Cleaning — Equipment Code Removal

Equipment codes (e.g., `P-1234`, `V-5678`, `T-9012`) are alphanumeric identifiers, not semantic content. If left in, they dominate the embedding space and cause similar equipment IDs to cluster together rather than similar maintenance issues.

**Example:** `"Pump P-1234 leak"` and `"Pump P-1235 leak"` should cluster on "leak", not on different pump IDs.

**Regex pattern:** `\b[A-Z]-\d{4,}\b` — matches single uppercase letter, hyphen, 4+ digits.

In [ ]:
def clean_text(text):
    """
    Remove equipment codes from maintenance descriptions.
    
    Equipment code format: [A-Z]-\d{4,}  (e.g., P-1234, V-5678)
    """
    text = re.sub(r'\b[A-Z]-\d{4,}\b', '', str(text))
    text = re.sub(r'\s+', ' ', text).strip()  # collapse whitespace
    return text

df['cleaned_description'] = df['DESCRIPTION'].apply(clean_text)

# Show before/after
before = "Pump P-1234 leak observed at flange V-5678"
after = clean_text(before)
print(f'Before: {before}')
print(f'After:  {after}')

## 5. Verify Cleaning Impact

Check that cleaning didn't destroy too much content.

In [ ]:
# Check description length before vs after cleaning
df['desc_len_before'] = df['DESCRIPTION'].apply(lambda x: len(str(x).split()))
df['desc_len_after'] = df['cleaned_description'].apply(lambda x: len(str(x).split()))

print('Average words per description:')
print(f"  Before cleaning: {df['desc_len_before'].mean():.1f}")
print(f"  After cleaning:  {df['desc_len_after'].mean():.1f}")
print(f"  Avg words removed: {df['desc_len_before'].mean() - df['desc_len_after'].mean():.1f}")

# Show some cleaned samples
print('\n=== Cleaned Samples ===')
for i, desc in enumerate(df['cleaned_description'].sample(3, random_state=42)):
    print(f"{i+1}. {desc}")

## 6. Stopword Removal (Experiment)

We experimented with stopword removal, but ultimately **kept stopwords in the final embeddings**. Why?

1. **Modern transformer context:** `all-MiniLM-L6-v2` was trained with stopwords present
2. **Grammatical context:** "pump is leaking" ≠ "pump leaking" — stopwords carry meaning
3. **Contextual embeddings:** Word meaning depends on surrounding words

The code below is documented for reproducibility but **not used** in the final pipeline.

In [ ]:
# ── Experimental: stopword removal (NOT used in final pipeline) ──

stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    """Remove English stopwords from text."""
    tokens = word_tokenize(text.lower())
    filtered = [word for word in tokens if word not in stop_words]
    return ' '.join(filtered)

# Demo the difference
sample = "The pump is leaking oil from the seal area"
print(f'Original:          {sample}')
print(f'Without stopwords: {remove_stopwords(sample)}')
print(f'\n➡ Decision: STOPWORDS KEPT (better for sentence transformers)')

## 7. Save Cleaned Dataset

Export the cleaned descriptions for use in subsequent notebooks.

In [ ]:
# Save the cleaned dataset
df.to_csv('../data/cleaned/dataset3_cleaned.csv', index=False)
print(f'✅ Saved cleaned dataset: {df.shape[0]} rows, {df.shape[1]} columns')
print(f'\nReady for Notebook 02 — Embedding Generation & Clustering')